**Summary:** We will explore data augmentation, as it relates to object detection tasks. We start by looking at the data being loaded by a trained YOLO model. This demonstrates that model is trained on augmented data, transformed from the the original images. We then use the Torchvision transforms to generate similar augmented data ourselves.

**Objectives:**

Explore the data loaded for training in the YOLO model

Observe augmented training data in YOLO

Use Torchvision transforms for data augmentation

See how bounding boxes need to transform along with images

Compose several augmentation techniques together

In [ ]:
!pip install ultralytics
!pip install torchinfo
!pip install torchvision
!pip install pillow

In [5]:
import pathlib
import sys

import matplotlib.pyplot as plt
import torch
import torchinfo
import torchvision
import ultralytics
from PIL import Image
from torchvision.transforms import v2
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("matplotlib version:", plt.matplotlib.__version__)
print("PIL version:", Image.__version__)
print("torch version:", torch.__version__)
print("torchvision version:", torchvision.__version__)
print("ultralytics version:", ultralytics.__version__)

Platform: linux
Python version: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
---
matplotlib version: 3.10.0
PIL version: 11.3.0
torch version: 2.8.0+cu126
torchvision version: 0.23.0+cu126
ultralytics version: 8.3.203


In [7]:
CLASS_DICT = dict(
    enumerate(
        [
            "ambulance",
            "army vehicle",
            "auto rickshaw",
            "bicycle",
            "bus",
            "car",
            "garbagevan",
            "human hauler",
            "minibus",
            "minivan",
            "motorbike",
            "pickup",
            "policecar",
            "rickshaw",
            "scooter",
            "suv",
            "taxi",
            "three wheelers (CNG)",
            "truck",
            "van",
            "wheelbarrow",
        ]
    )
)

print("CLASS_DICT type,", type(CLASS_DICT))
CLASS_DICT

CLASS_DICT type, <class 'dict'>


{0: 'ambulance',
 1: 'army vehicle',
 2: 'auto rickshaw',
 3: 'bicycle',
 4: 'bus',
 5: 'car',
 6: 'garbagevan',
 7: 'human hauler',
 8: 'minibus',
 9: 'minivan',
 10: 'motorbike',
 11: 'pickup',
 12: 'policecar',
 13: 'rickshaw',
 14: 'scooter',
 15: 'suv',
 16: 'taxi',
 17: 'three wheelers (CNG)',
 18: 'truck',
 19: 'van',
 20: 'wheelbarrow'}

Earlier, you trained a YOLO model with some images.

You might think YOLO just used your images exactly as they are, but actually it changes them a bit during training (this is where data augmentation comes in).

To show what really happened to those images, you’re going to reload the saved model and look inside it.

The first step is to find where your saved training runs (the model checkpoints) are stored.

**👉 In short:**
The purpose of this paragraph is to explain that YOLO doesn’t just use your raw images—it applies transformations (like augmentation) during training. Now, you’ll reload the model to see proof of that.

**Task 3.5.1: Choose a training run, and check that there are model weights saved in the weights/best.pt file for that run.**

In [ ]:
run_dir = runs_dir / "train"
weights_file = run_dir / "weights" / "best.pt"

print("Weights file exists?", weights_file.exists())

**Task 3.5.2: Load the model from the weights file.**

In [ ]:
model = YOLO(weights_file)
torchinfo.summary(model)

We need to get the model set up to load the data. The easiest way to do that is to train it for an epoch.

T**ask 3.5.3: Run one epoch of training.**

In [ ]:
result = model.train(
    data=model.overrides["data"],
    epochs=1,
    batch=8,
    workers=1,
)

The model should now have a .trainer attribute, which has a .train_loader attribute. This will be a DataLoader that loads the training data.







**Task 3.5.4: Save this data loader to variable loader.**

In [ ]:
loader = model.trainer.train_loader

print(type(loader))

Data loaders are iterables. That is, you can put them in a for loop to load data one batch at a time. We just want to read one batch from it, though.

**Task 3.5.5: Load one batch from loader into the variable batch. You can do this by constructing a for loop over loader and calling break inside the loop, so that it only runs once.**

In [ ]:
for batch in loader:
    break

print(type(batch))

We get back a dictionary. (What a surprise!) Let's explore what's in this structure.







Task 3.5.6: Print out the keys in batch **bold text**

In [ ]:
print(batch.keys())

**Task 3.5.7: Print out the shape of the img value.**

In [ ]:
print(batch['img'].shape)

The dimension of 3 represents the color channels. The dimension of 640 are the width and height. So what does the dimension of 8 represent?

You can get a clue by reviewing the call to model.train. We set a batch size of 8. This tensor thus represents eight training images.

**Task 3.5.8: Print out the shape of the bboxes value.**

In [ ]:
print(batch['bboxes'].shape)


**Task 3.5.9: Plot the image and bounding boxes for index 0 of this batch.**

In [ ]:
def plot_with_bboxes(img, bboxes, cls, batch_idx=None, index=0, **kw):
    """Plot the bounding boxes on an image.

    Input:  img     The image, either as a 3-D tensor (one image) or a
                    4-D tensor (a stack of images).  In the latter case,
                    the index argument specifies which image to display.
            bboxes  The bounding boxes, as a N x 4 tensor, in normalized
                    XYWH format.
            cls     The class indices associated with the bounding boxes
                    as a N x 1 tensor.
            batch_idx   The index of each bounding box within the stack of
                        images.  Ignored if img is 3-D.
            index   The index of the image in the stack to be displayed.
                    Ignored if img is 3-D.
            **kw    All other keyword arguments are accepted and ignored.
                    This allows you to use dictionary unpacking with the
                    values produced by a YOLO DataLoader.
    """
    if img.ndim == 3:
        image = img[None, :]
        index = 0
        batch_idx = torch.zeros((len(cls),))
    elif img.ndim == 4:
        # Get around Black / Flake8 disagreement
        indp1 = index + 1
        image = img[index:indp1, :]

    inds = batch_idx == index
    res = ultralytics.utils.plotting.plot_images(
        images=image,
        batch_idx=batch_idx[inds] - index,
        cls=cls[inds].flatten(),
        bboxes=bboxes[inds],
        names=CLASS_DICT,
        threaded=False,
        save=False,
    )

    return Image.fromarray(res)

In [ ]:
plot_with_bboxes(**batch, index=0)

That's ... weird looking. It's not what our original images look like, is it?

If it's not weird looking, try looking at another index. Eventually you'll find something weird looking!
The file names from the batch are stored in the im_file key. We can use that to look up the original image associated with this index and see what it looks like.






**Task 3.5.10: Display the original image file for this index.**

In [ ]:
Image.open(batch['im_file'][0])

Comparing the two, we can see that the original image was distorted and combined with other images before being loaded into the YOLO model. The YOLO model applies a number of augmentation steps by default. (You can take a look at all of the augmentation settings in YOLO.) This increases the diversity of training images, which should help the model generally.

Data Augmentation with Torchvision
If you're training a YOLO model, it's generally best to use the built-in augmentation setting. But in other cases, you may need to implement an augmentation system yourself. Torchvision makes this easy by providing a number of augmentation transforms in its transforms version 2 (v2) module.

To demonstrate this, we'll load a sample image. The code below will get the file paths for 01.jpg and its associated label file. (It's written so that it works whether the image ended up in the training or validation split.)

In [ ]:
yolo_base = pathlib.Path("data_yolo")
sample_fn = next((yolo_base / "images").glob("*/01.jpg"))
sample_labels = next((yolo_base / "labels").glob("*/01.txt"))

print(sample_fn)
print(sample_labels)

**Task 3.5.11: Load the image with PIL.**

In [ ]:
sample_image = Image.open(sample_fn)

sample_image

The bounding boxes are stored in the label file. Let's take a look a the first five lines to remember what it looks like.

!head -n5 $sample_labels

**Task 3.5.12: Convert the image to a tensor. In the transforms version 2 module, this can be done with the confusingly-named ToImage transform.**

In [ ]:
sample_torch = v2.ToImage()(sample_image)

print(sample_torch.shape)

Each line represents a bounding box. The first element is the class index. This is followed by the x and y coordinates of the box center, the width, and the height.

**Task 3.5.13: Load the bounding box data into a variable named label_data. It should be a list of the bounding boxes. Each bounding box will itself be a list of five strings in the same order they are in the file. Don't worry about converting the strings to numbers yet.**

In [ ]:
# Load the data into `label_data`
with open(sample_labels, "r") as file:
     label_data=[row.split() for row in file]

label_data[:5]

**Task 3.5.14: Create a tensor containing the class indices. For compatibility with our plotting function it should be a tensor.**

In [ ]:
classes = torch.Tensor([int(row[0]) for row in label_data])

print("Tensor shape:", classes.shape)
print("First 5 elements:\n", classes[:5])

**Task 3.5.15: Load the bounding box coordinates into a tensor.**

In [ ]:
bboxes =torch.Tensor([[float(element) for element in row[1:]] for row in label_data])
print("Tensor shape:", bboxes.shape)
print("First 5 elements:\n", bboxes[:5])

**Task 3.5.16: Convert the bounding box coordinates to pixel units.**

In [ ]:
sample_width, sample_height = sample_image.size

scale_factor = torch.Tensor([sample_width, sample_height,
                        sample_width,sample_height])

bboxes_pixels = bboxes * scale_factor

print("Tensor shape:", bboxes_pixels.shape)
print("First 5 elements:\n", bboxes_pixels[:5])


In order for the transformations to know how to transform the bounding boxes, they need to know that the coordinates represent the centers and dimensions of the boxes. This is done by creating a special BoundingBoxes tensor. This type has a format attribute. By setting this to "CXCYWH", we're telling it that the columns represent the Center X coordinate, the Center Y coordinate, the Width, and the Height. This tensor also is given the size of the image, so it doesn't need to look that up for transformations.

In [ ]:
bboxes_tv = torchvision.tv_tensors.BoundingBoxes(
    bboxes_pixels,
    format="CXCYWH",
    # Yes, that's right.  Despite using width x height everywhere
    # else, here we have to specify the image size as height x
    # width.
    canvas_size=(sample_height, sample_width),
)

print("Tensor type:", type(bboxes_tv))
print("First 5 elements:\n", bboxes_tv[:5])

Let's double check that we did all of those conversions correctly. Do the bounding boxes line up with the correct objects?

In [ ]:
plot_with_bboxes(sample_torch, bboxes, classes)

If everything looks good, we'll introduce some transformations. The first one will be a horizontal flip. Many everyday objects have bilateral symmetry (or nearly so), so a flipped image will still have the same object classes in it. This makes a horizontal flip a good data augmentation transformation.

(In contrast, up/down symmetry is less common. A vertical flip is generally not as useful, unless you need to recognize upside-down objects.)

The transforms version 2 module has a RandomHorizontalFlip transformation. This takes the probability of a flip as an argument.

**Task 3.5.17: Use the RandomHorizontalFlip transformation to flip the sample image. Set p=1 to ensure that the flip happens.**

In [ ]:
flipped = v2.RandomHorizontalFlip(p=1)(sample_torch)

plot_with_bboxes(flipped, bboxes_tv, classes)

The image has flipped, but the bounding boxes are still in their original locations. Note that the bus is now in the bottom left of the image. Its bounding box is still at the bottom right, and it now contains some asphalt, planters and trees. If we fed this into a model, it would make the model worse, by confusing it as to what a bus looks like.

So, we need to transform the bounding box coordinates consistent with the image transformation. The Torchvision version 2 transformations can take multiple arguments. They perform the same transformation on all of the arguments, returning a transformed version of each. They also understand how to correctly transform the BoundingBoxes tensors, depending on their type.

**Task 3.5.18: Use RandomHorizontalFlip to flip both the sample image and its bounding boxes. Check that they line up correctly now.**

In [ ]:
flipped, flipped_bboxes = v2.RandomHorizontalFlip(p=1)(sample_torch,bboxes_tv)

plot_with_bboxes(flipped, flipped_bboxes, classes)

**Task 3.5.19: Apply the RandomRotation transformation. This takes an argument of the maximum number of degrees to rotate the image. Set it to 90.**

In [ ]:
rotated, rotated_bboxes = v2.RandomRotation(90)(sample_torch, bboxes_tv)

plot_with_bboxes(rotated, rotated_bboxes, classes)

**Task 3.5.20: Create an augmentation pipeline that combines the RandomHorizontalFlip with the RandomRotation. This time, set the probability of the flip to 50%.**

In [ ]:
transforms = v2.Compose(
    [
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomRotation(degrees=90)
    ]
)

transformed, transformed_bboxes = transforms(sample_torch,bboxes_tv)

plot_with_bboxes(transformed, transformed_bboxes, classes)

There are a large number of transformations that can be used for data augmentation. Scroll through the documentation to get a view of the range of possibilities.

In addition to the transforms we've already used, note:

RandomResizedCrop will randomly crop the image down, and then it resizes the output to a set dimension.
ColorJitter can randomly adjust the brightness, contrast, saturation, and hue of the image, within specified ranges.


**Task 3.5.21: Create an augmentation pipeline that applies several of these transformations. Choose reasonable values for the parameters. Check that the bounding boxes are transformed correctly through this.**

In [ ]:
transforms = v2.Compose(
    [
        v2.RandomResizedCrop(size=(640, 640)),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomRotation(degrees=30),
        v2.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
    ]
)

transformed, transformed_bboxes = transforms(sample_torch, bboxes_tv)
plot_with_bboxes(transformed, transformed_bboxes, classes)

You can run the transformation several times to see the different types of images that result. This greater diversity of training images will help models learn to generalize instead of memorizing during training.